In [1]:
from pathlib import Path
import pandas as pd
import importlib, backtesting.data as btdata
importlib.reload(btdata)


from backtesting.main import compute_start_end_dates
from backtesting.data import fetch_sp500_universe, download_price_history, fetch_sp500_universe_cached
from backtesting.strategies import OneYearMomentum
from backtesting.engine import run_backtest
from backtesting.reporting import compile_performance_tables
from backtesting.plotting import plot_cumulative_performance

years = 10
start, end = compute_start_end_dates(years)

universe = fetch_sp500_universe_cached("data/sp500_universe.csv", refresh=False)

tickers_for_prices = list(dict.fromkeys(universe.tickers + [universe.benchmark]))  # add SPY
prices = download_price_history(
    tickers_for_prices,
    start=start,
    end=end,
    monthly=True,
    refresh=False,
    limit=120,        # warm cache first; remove later
    source="stooq",
)
print(prices.shape, prices.columns[:10])

strategy = OneYearMomentum(top_n=50)
weights  = strategy.generate_weights(prices)
result   = run_backtest(universe, prices, weights, start=start, end=end)

tables = compile_performance_tables(result.strategy_returns, result.benchmark_returns, rf_annual=0.0)
display(tables["summary_fmt"])
display(tables["yearly_fmt"])

cumulative = pd.DataFrame({
    "Momentum": (1 + result.strategy_returns).cumprod(),
    "S&P 500": (1 + result.benchmark_returns).cumprod(),
})

plot_path = plot_cumulative_performance(cumulative, Path("output/cumulative_performance.png"))
plot_path

(134, 120) Index(['A', 'AAPL', 'ABBV', 'ABNB', 'ABT', 'ACGL', 'ACN', 'ADBE', 'ADI',
       'ADM'],
      dtype='object')
[download_price_history] Only 1 tickers succeeded (<25). Re-run to fill cache or set source='stooq'.


,CAGR,Volatility,Max Drawdown,Sharpe
Strategy,16%,15%,-20%,1.060817
Benchmark,13%,15%,-24%,0.914836


,Strategy,Benchmark
Year,,
2014,5%,4.9%
2015,3.1%,1.2%
2016,13%,12%
2017,35%,22%
2018,-0.76%,-4.6%
2019,35%,31%
2020,21%,18%
2021,26%,29%
2022,-5%,-18%


PosixPath('output/cumulative_performance.png')